# Sim 1: Forward kinematics, with the simulator as referee

**Roles today.** The **Driver** says the next step out loud. The **Keyboard**
types only what someone else has said. The **Checker** owns the board.

**The rule.** Every cell marked ▶ RUN is preceded by a 🔮 PREDICT. Before
anyone runs it, the Checker writes the group's prediction on the board. After
it runs, the group says what matched, what did not, and why, before moving on.

**The clock.** Part A by 0:15, B1 by 0:35, B2 by 0:55, B3 by 1:05, then Part C
together. If you are behind, skip to Part C when the instructor calls it; the
submission finishes what you started.

CoppeliaSim must be open before you run the first cell.


In [ ]:
import numpy as np
import screws as sc
from me465 import close, open_lab, settle

np.set_printoptions(precision=4, suppress=True)
scene, arm = open_lab()
print(arm)

## Part A: Warm-up (every group in step)

The notes give MR's model of the UR5: its home configuration $M$ and its
space-frame screw axes $\mathcal{S}_1, \dots, \mathcal{S}_6$ (MR Example 4.5),
from the rounded dimensions of the textbook's figure. They are typed in for you.

**Check one row.** Before running, pick a joint and confirm its
$\omega_i$ and $v_i = -\omega_i \times q_i$ against the notes' table.


In [ ]:
W1, W2, L1, L2, H1, H2 = 0.109, 0.082, 0.425, 0.392, 0.089, 0.095

M_MR = np.array([[-1, 0, 0, L1 + L2],
                 [ 0, 0, 1, W1 + W2],
                 [ 0, 1, 0, H1 - H2],
                 [ 0, 0, 0, 1]])

# one screw axis per row, S_i = (omega_i, v_i) with v_i = -omega_i x q_i; the .T makes them columns
S_MR = np.array([sc.revolute_axis(q=[0, 0, 0],            s_hat=[0, 0, 1]),
                 sc.revolute_axis(q=[0, 0, H1],           s_hat=[0, 1, 0]),
                 sc.revolute_axis(q=[L1, 0, H1],          s_hat=[0, 1, 0]),
                 sc.revolute_axis(q=[L1 + L2, 0, H1],     s_hat=[0, 1, 0]),
                 sc.revolute_axis(q=[L1 + L2, W1, 0],     s_hat=[0, 0, -1]),
                 sc.revolute_axis(q=[L1 + L2, 0, H1 - H2], s_hat=[0, 1, 0])]).T
print(S_MR)


🔮 **PREDICT.** We are about to command every joint to $\theta_i = 0$. Sketch
on the board what the robot will look like, and write down where you expect
the tool frame to be.

The robot stands on a pedestal 0.4 m tall, and MR's $\{s\}$ sits on top of it.
So the world frame and $\{s\}$ differ by a pure lift:
$T_{ws} = \mathrm{Trans}(0, 0, 0.4)$.

In [ ]:
T_ws = sc.rp_to_transform(np.eye(3), [0, 0, 0.4])
T_sw = sc.transform_inv(T_ws)

# ▶ RUN
arm.command_positions(np.zeros(6))
settle(scene, arm)
T_s_tool = T_sw @ arm.tip_frame()
print("tool at theta = 0, in {s}:\n", T_s_tool)
print("M_MR:\n", M_MR)

📺 **Screens up** when the instructor calls it. Everyone's robot should look
the same. Does it look like your sketch? Does `T_s_tool` look like `M_MR`?

## Part B: Build

### B1. Read the screw axes off the simulated robot

PoE works from *any* home configuration, as long as $M$ and the screw axes
are measured in that same configuration. So let us take the scene's own zero
pose as the home, and read $M$ and every $\mathcal{S}_i$ off the simulator.

`arm.joint_frames()` gives each joint's frame in the world frame. A joint turns
about its frame's $z$-axis, so for joint $i$, with $T_{sj}$ its frame
expressed in $\{s\}$:

- $\hat\omega_i$ is the third column of $R_{sj}$
- $q_i$, any point on the axis, can be the frame's origin $p_{sj}$
- $v_i = -\hat\omega_i \times q_i$

🔮 **PREDICT.** Before you run it: which columns of `S_scene` will match
`S_MR`, and which will not? Why?

In [ ]:
arm.command_positions(np.zeros(6))
settle(scene, arm)

S_cols = []
for T_wj in arm.joint_frames():
    T_sj = T_sw @ T_wj
    # TODO: omega, q, and v for this joint, then append the 6-vector
    pass
S_scene = np.array(S_cols).T
M_scene = T_sw @ arm.tip_frame()

# ▶ RUN
print("S_scene:\n", S_scene)
print("M_scene:\n", M_scene)

### B2. Your configuration card

Your group's card gives six joint angles in degrees. For the card:

1. 🔮 **PREDICT, roughly.** Sketch the arm on the board. Where is the gripper?
2. 🔮 **PREDICT, exactly.** Compute $T_{s,\text{tool}}$ with
   `sc.fk_space(M_scene, S_scene, theta)` (the same function as
   MR's `FKinSpace`, under its `screws` name).
3. ▶ **RUN.** Command the angles, let the arm settle, and read the ground truth.
4. **Explain the gap.**

In [ ]:
card = [0, 0, 0, 0, 0, 0]   # TODO: your card, in degrees
theta = np.radians(card)

T_pred = ...                # TODO: forward kinematics with the scene's model

# ▶ RUN
arm.command_positions(theta)
measured = settle(scene, arm)
T_true = T_sw @ arm.tip_frame()


def pose_error(T_a, T_b):
    """Position error in mm, rotation error in degrees."""
    dp = np.linalg.norm(T_a[:3, 3] - T_b[:3, 3]) * 1000
    R = T_a[:3, :3].T @ T_b[:3, :3]
    dth = np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1)))
    return dp, dth


print("predicted:\n", T_pred, "\nsimulator:\n", T_true)
print("error: %.2f mm, %.3f deg" % pose_error(T_pred, T_true))
print("measured angles minus commanded (deg):", np.degrees(measured - theta))

### B3. The book's model, driven through the offset

In MR's home configuration the arm is stretched out level. The scene's zero
pose is not: it stands the arm straight up. The six angles that put the
simulated arm in MR's home pose are a fact about the scene, given here:

$$\theta_\text{offset} = (-90^\circ, 90^\circ, 0, 90^\circ, 0, 90^\circ).$$

So a joint vector for MR's model and one for the scene are related by
$\theta_\text{scene} = \theta_\text{MR} + \theta_\text{offset}$, and MR's model at
$\theta_\text{MR}$ should land where the simulator puts the robot at
$\theta_\text{scene}$.

🔮 **PREDICT.** First for the home pose itself: run the offset alone. How
close to `M_MR` will the tool be, in millimetres and degrees? Then for your
card: how close will MR's model come to the simulator?


In [ ]:
offset = np.radians([-90, 90, 0, 90, 0, 90])

# ▶ RUN: the home pose
arm.command_positions(offset)
settle(scene, arm)
T_home = T_sw @ arm.tip_frame()
print("simulator at the offset:\n", T_home, "\nM_MR:\n", M_MR)
print("error: %.2f mm, %.3f deg" % pose_error(T_home, M_MR))

# ▶ RUN: your card, through MR's model
theta_MR = theta - offset
T_book = ...    # TODO: MR's model at theta_MR
arm.command_positions(theta)
settle(scene, arm)
print("card, book vs simulator: %.2f mm, %.3f deg" % pose_error(T_book, T_sw @ arm.tip_frame()))


When the orientation matches, look at what is left in the position: a few
millimetres, always in the same direction. Is that a mistake, or are two
models of the same robot really different? Say on the board which
dimension you suspect, and how you would find out. The submission asks you
to.


## Part C: Converge

Wait for the instructor. Every group commands the same angles at the same
moment, and we compare the robots side by side.

In [ ]:
converge = np.radians([0, 0, 0, 0, 0, 0])   # the instructor gives these in class

# ▶ RUN
arm.command_positions(converge)
settle(scene, arm)
print("tool, in {s}:\n", T_sw @ arm.tip_frame())

When you are done for the day:

In [ ]:
close(scene)